# 03 — Avaliação LLM (Judge + RAGAS)

Análise consolidada dos resultados de avaliação do agente.
- **Judge**: 3 critérios de negócio (factual, relevância, hallucination)
- **RAGAS**: 4 métricas do RAG (faithfulness, relevancy, precision, recall)
- **A/B Test**: Comparação de prompts
- **Benchmark**: Comparação de modelos LLM

In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

# Diretório de resultados
results_dir = Path("../evaluation/results")
print(f"📁 Resultados em: {results_dir.resolve()}")
print(f"   Arquivos: {list(results_dir.glob('*'))[:5]}...")

📁 Resultados em: /Users/alan/Data Science Projects/datathon-techchallenge-fase-5/evaluation/results
   Arquivos: [PosixPath('../evaluation/results/ragas_responses.jsonl'), PosixPath('../evaluation/results/judge_responses.jsonl'), PosixPath('../evaluation/results/llm_judge.json')]...


## 1. Load dos Resultados

In [2]:
# Judge: respostas individuais
judge_responses = []
if (results_dir / "judge_responses.jsonl").exists():
    with open(results_dir / "judge_responses.jsonl") as f:
        for line in f:
            judge_responses.append(json.loads(line))
    print(f"✓ Judge: {len(judge_responses)} respostas carregadas")
else:
    print("⚠ judge_responses.jsonl não encontrado")

# Judge: resumo final
judge_summary = {}
if (results_dir / "llm_judge.json").exists():
    with open(results_dir / "llm_judge.json") as f:
        judge_summary = json.load(f)
    print(f"✓ Judge Summary carregado")

# RAGAS: respostas individuais
ragas_responses = []
if (results_dir / "ragas_responses.jsonl").exists():
    with open(results_dir / "ragas_responses.jsonl") as f:
        for line in f:
            ragas_responses.append(json.loads(line))
    print(f"✓ RAGAS: {len(ragas_responses)} respostas carregadas")
else:
    print("⚠ ragas_responses.jsonl não encontrado")

# RAGAS: resumo final
ragas_summary = {}
if (results_dir / "ragas_eval.json").exists():
    with open(results_dir / "ragas_eval.json") as f:
        ragas_summary = json.load(f)
    print(f"✓ RAGAS Summary carregado")

✓ Judge: 21 respostas carregadas
✓ Judge Summary carregado
✓ RAGAS: 21 respostas carregadas


In [3]:
# A/B Test (APÓS RUN DE A/B TESTS)
ab_test_results = None
if (results_dir / "ab_test_output.json").exists():
    with open(results_dir / "ab_test_output.json") as f:
        ab_test_results = json.load(f)
    print(f"✓ A/B Test carregado (vencedor: {ab_test_results.get('winner', 'N/A')})")
else:
    print("⚠ ab_test_output.json não encontrado (ainda não executado)")

# Benchmark (APÓS RUN DE BENCHMARK)
benchmark_results = None
if (results_dir / "benchmark_output.json").exists():
    with open(results_dir / "benchmark_output.json") as f:
        benchmark_results = json.load(f)
    print(f"✓ Benchmark carregado ({len(benchmark_results.get('results', []))} configs)")
else:
    print("⚠ benchmark_output.json não encontrado (ainda não executado)")

⚠ ab_test_output.json não encontrado (ainda não executado)
⚠ benchmark_output.json não encontrado (ainda não executado)


## 2. Tabela Principal (Judge + RAGAS)

In [4]:
judge_responses

[{'pergunta': 'Qual a previsão do IBOV para o próximo pregão?',
  'answer': 'A previsão do IBOV para o próximo pregão é de 180,291.61 pontos, com uma variação prevista de -2.07% em relação ao último fechamento de 184,108.00. É importante lembrar que a acurácia direcional histórica do modelo univariado é de aproximadamente 50%, portanto, essa previsão deve ser usada como um filtro de viés e não como um sinal autônomo.',
  'factual_correctness': 5,
  'business_relevance': 5,
  'no_hallucination': 5,
  'rationale': 'A resposta fornece números específicos e uma variação prevista, além de incluir um disclaimer sobre a acurácia direcional, demonstrando conhecimento do ground truth e relevância para o analista de mercado.'},
 {'pergunta': 'Qual a expectativa do mercado para a Selic ao final de 2026 segundo o Focus mais recente?',
  'answer': 'Agent stopped due to iteration limit or time limit.',
  'factual_correctness': 1,
  'business_relevance': 1,
  'no_hallucination': 5,
  'rationale': 'A 

In [5]:
# Consolidar Judge + RAGAS em uma tabela única
data = []

for judge_row in judge_responses:
    q = judge_row["pergunta"]
    
    # Encontrar a resposta equivalente em RAGAS
    ragas_row = next((r for r in ragas_responses if r["question"] == q), None)
    
    row = {
        "Pergunta": q,
        "Resposta": judge_row.get("answer",0),
        "Judge_Factual": judge_row.get("factual_correctness", 0),
        "Judge_Relevância": judge_row.get("business_relevance", 0),
        "Judge_NoHalluc": judge_row.get("no_hallucination", 0),
        "Judge_Média": np.mean([judge_row.get("factual_correctness", 0),
                                 judge_row.get("business_relevance", 0),
                                 judge_row.get("no_hallucination", 0)]),
        "Razão": judge_row.get("rationale",0)
    }
    
    # Adicionar RAGAS se disponível (vai ser adicionado depois de compilar metrics)
    if ragas_row:
        row["RAGAS_Faithful"] = ragas_row.get("faithfulness", 0)
        row["RAGAS_Relevant"] = ragas_row.get("answer_relevancy", 0)
    
    data.append(row)

df_results = pd.DataFrame(data)

# Exibir tabela
print(f"\n📊 Tabela de Resultados ({len(df_results)} perguntas)\n")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
display(df_results.round(2))


📊 Tabela de Resultados (21 perguntas)



,Pergunta,Resposta,Judge_Factual,Judge_Relevância,Judge_NoHalluc,Judge_Média,Razão,RAGAS_Faithful,RAGAS_Relevant
0,Qual a previsão do IBOV para o próximo pregão?,A previsão do IBOV para o próximo pregão é de ...,5,5,5,5.00,A resposta fornece números específicos e uma v...,0,0
1,Qual a expectativa do mercado para a Selic ao ...,Agent stopped due to iteration limit or time l...,1,1,5,2.33,A resposta não fornece informações relevantes ...,0,0
2,"Se o IBOV passar de 128.500 para 130.300, qual...",A variação percentual do IBOV seria de aproxim...,5,5,5,5.00,A resposta está correta e é útil para um anali...,0,0
3,Como o dólar está hoje e qual a relação com a ...,"O dólar está atualmente em 4.89, com uma varia...",4,4,4,4.00,A resposta fornece informações relevantes sobr...,0,0
4,Houve alguma decisão recente do Copom que poss...,A decisão recente do Copom de reduzir a taxa b...,4,4,5,4.33,"A resposta está próxima do ground truth, mas n...",0,0
5,Qual foi a taxa de crescimento do PIB brasilei...,A taxa de crescimento projetada para o Produto...,5,4,5,4.67,A resposta está correta de acordo com o ground...,0,0
6,Quais são as projeções de crescimento do PIB p...,"A projeção de crescimento do PIB para 2026, se...",4,3,5,4.00,"A resposta está correta em termos de valor, ma...",0,0
7,"De acordo com o Morning Call da XP, qual foi a...",Agent stopped due to iteration limit or time l...,0,0,0,0.00,A resposta não fornece nenhuma informação fact...,0,0
8,Qual foi a justificativa do Federal Reserve pa...,Agent stopped due to iteration limit or time l...,0,0,0,0.00,A resposta não fornece nenhuma informação rele...,0,0
9,O que foi decidido pelo Senado em relação à in...,O Senado rejeitou a indicação de Jorge Messias...,5,2,5,4.00,A resposta está correta em relação ao ground t...,0,0


## 3. Resumo Simples

In [6]:
print("\n📈 RESUMO DAS MÉTRICAS\n")
print("="*70)

if judge_summary.get("averages"):
    avg = judge_summary["averages"]
    print("\n🔍 JUDGE (escala 1-5)")
    print(f"  Factual Correctness:  {avg.get('factual_correctness_avg', 0):.2f}/5.0")
    print(f"  Business Relevance:   {avg.get('business_relevance_avg', 0):.2f}/5.0")
    print(f"  No Hallucination:     {avg.get('no_hallucination_avg', 0):.2f}/5.0")
    print(f"  Média Geral:          {np.mean([avg.get('factual_correctness_avg', 0), avg.get('business_relevance_avg', 0), avg.get('no_hallucination_avg', 0)]):.2f}/5.0")

if ragas_summary:
    print("\n📊 RAGAS (escala 0-1)")
    print(f"  Faithfulness:         {ragas_summary.get('faithfulness', 0):.3f}")
    print(f"  Answer Relevancy:     {ragas_summary.get('answer_relevancy', 0):.3f}")
    print(f"  Context Precision:    {ragas_summary.get('context_precision', 0):.3f}")
    print(f"  Context Recall:       {ragas_summary.get('context_recall', 0):.3f}")
    print(f"  Média Geral:          {np.mean([ragas_summary.get('faithfulness', 0), ragas_summary.get('answer_relevancy', 0), ragas_summary.get('context_precision', 0), ragas_summary.get('context_recall', 0)]):.3f}")

print("\n" + "="*70)


📈 RESUMO DAS MÉTRICAS


🔍 JUDGE (escala 1-5)
  Factual Correctness:  2.43/5.0
  Business Relevance:   2.43/5.0
  No Hallucination:     3.33/5.0
  Média Geral:          2.73/5.0



In [7]:
# % de perguntas com score alto
print("\n✅ DISTRIBUIÇÃO DE SCORES (Judge)\n")

if "Judge_Média" in df_results.columns:
    scores = df_results["Judge_Média"]
    print(f"  Score ≥ 4.5/5 (Excelente): {(scores >= 4.5).sum()} perguntas ({(scores >= 4.5).sum()/len(scores)*100:.0f}%)")
    print(f"  Score 3.5-4.5 (Bom):       {((scores >= 3.5) & (scores < 4.5)).sum()} perguntas ({((scores >= 3.5) & (scores < 4.5)).sum()/len(scores)*100:.0f}%)")
    print(f"  Score < 3.5 (Precisa melhora): {(scores < 3.5).sum()} perguntas ({(scores < 3.5).sum()/len(scores)*100:.0f}%)")


✅ DISTRIBUIÇÃO DE SCORES (Judge)

  Score ≥ 4.5/5 (Excelente): 5 perguntas (24%)
  Score 3.5-4.5 (Bom):       4 perguntas (19%)
  Score < 3.5 (Precisa melhora): 12 perguntas (57%)


## 4. Problemas Identificados

In [8]:
print("\n⚠️  PERGUNTAS COM BAIXA PERFORMANCE (Judge < 3.5)\n")

if "Judge_Média" in df_results.columns:
    problematic = df_results[df_results["Judge_Média"] < 3.5].sort_values("Judge_Média")

    if len(problematic) > 0:
        for idx, row in problematic.iterrows():
            print(f"\n{'='*100}")
            print(f"❌ PERGUNTA:")
            print(f"   {row['Pergunta']}")
            print(f"\n📝 RESPOSTA DO AGENTE:")
            if "Resposta" in row and pd.notna(row["Resposta"]):
                print(f"   {row['Resposta']}")
            else:
                print("   (Resposta não disponível)")
            print(f"\n📊 SCORES:")
            print(f"   Judge Score Geral: {row['Judge_Média']:.2f}/5.0")
            print(f"   • Factual Correctness: {row['Judge_Factual']}/5")
            print(f"   • Business Relevance: {row['Judge_Relevância']}/5")
            print(f"   • No Hallucination: {row['Judge_NoHalluc']}/5")
            if "RAGAS_Faithful" in row and not pd.isna(row["RAGAS_Faithful"]):
                print(f"   • RAGAS Faithfulness: {row['RAGAS_Faithful']:.3f}")
            if "RAGAS_Relevant" in row and not pd.isna(row["RAGAS_Relevant"]):
                print(f"   • RAGAS Relevancy: {row['RAGAS_Relevant']:.3f}")
            print(f"\n🤔 RAZÃO")
            print(f"   {row['Razão']}")
            print(f"{'='*100}")
    else:
        print("✅ Nenhuma pergunta com score baixo!")
else:
    print("(Dados não disponíveis)")


⚠️  PERGUNTAS COM BAIXA PERFORMANCE (Judge < 3.5)


❌ PERGUNTA:
   De acordo com o Morning Call da XP, qual foi a decisão do Copom sobre a Selic na reunião de abril de 2026?

📝 RESPOSTA DO AGENTE:
   Agent stopped due to iteration limit or time limit.

📊 SCORES:
   Judge Score Geral: 0.00/5.0
   • Factual Correctness: 0/5
   • Business Relevance: 0/5
   • No Hallucination: 0/5
   • RAGAS Faithfulness: 0.000
   • RAGAS Relevancy: 0.000

🤔 RAZÃO
   A resposta não fornece nenhuma informação factual correta ou relevante para um analista de mercado, e sim indica um erro de processamento.

❌ PERGUNTA:
   Qual foi a justificativa do Federal Reserve para manter os juros em abril de 2026?

📝 RESPOSTA DO AGENTE:
   Agent stopped due to iteration limit or time limit.

📊 SCORES:
   Judge Score Geral: 0.00/5.0
   • Factual Correctness: 0/5
   • Business Relevance: 0/5
   • No Hallucination: 0/5
   • RAGAS Faithfulness: 0.000
   • RAGAS Relevancy: 0.000

🤔 RAZÃO
   A resposta não fornece nenhuma in

In [9]:
# Discordâncias entre Judge e RAGAS
print("\n🔀 DISCORDÂNCIAS (Judge vs RAGAS)\n")

if "Judge_Média" in df_results.columns and "RAGAS_Faithful" in df_results.columns:
    # Normalizar para 0-1
    df_results["Judge_Norm"] = df_results["Judge_Média"] / 5.0
    df_results["Discrepância"] = abs(df_results["Judge_Norm"] - df_results["RAGAS_Faithful"])
    
    discrepant = df_results[df_results["Discrepância"] > 0.3].sort_values("Discrepância", ascending=False)
    
    if len(discrepant) > 0:
        print(f"Encontradas {len(discrepant)} perguntas com discordância > 0.3:\n")
        for idx, row in discrepant.iterrows():
            print(f"  • {row['Pergunta'][:55]}...")
            print(f"    Judge: {row['Judge_Média']:.2f}/5 | RAGAS: {row['RAGAS_Faithful']:.3f} | Delta: {row['Discrepância']:.3f}")
    else:
        print("✅ Métodos bem alinhados (discordância < 0.3)")
else:
    print("(Dados não disponíveis)")


🔀 DISCORDÂNCIAS (Judge vs RAGAS)

Encontradas 15 perguntas com discordância > 0.3:

  • Qual a previsão do IBOV para o próximo pregão?...
    Judge: 5.00/5 | RAGAS: 0.000 | Delta: 1.000
  • Se o IBOV passar de 128.500 para 130.300, qual a variaç...
    Judge: 5.00/5 | RAGAS: 0.000 | Delta: 1.000
  • Segundo a ata do Copom de março de 2020, quais foram os...
    Judge: 5.00/5 | RAGAS: 0.000 | Delta: 1.000
  • Qual foi a taxa de crescimento do PIB brasileiro em 202...
    Judge: 4.67/5 | RAGAS: 0.000 | Delta: 0.933
  • Como a decisão da Suprema Corte dos EUA afetou a políti...
    Judge: 4.67/5 | RAGAS: 0.000 | Delta: 0.933
  • Houve alguma decisão recente do Copom que possa explica...
    Judge: 4.33/5 | RAGAS: 0.000 | Delta: 0.867
  • Como o dólar está hoje e qual a relação com a previsão ...
    Judge: 4.00/5 | RAGAS: 0.000 | Delta: 0.800
  • Quais são as projeções de crescimento do PIB para o ano...
    Judge: 4.00/5 | RAGAS: 0.000 | Delta: 0.800
  • O que foi decidido pelo Senado e

## 5. A/B Test (Prompts)

In [10]:
if ab_test_results:
    print("\n🏆 A/B TEST DE PROMPTS\n")
    print(f"Vencedor (Faithfulness): {ab_test_results.get('winner', 'N/A')}\n")
    
    results_df = pd.DataFrame(ab_test_results["results"])
    display(results_df[["prompt_name", "faithfulness", "answer_relevancy", "context_precision", "context_recall"]].round(3))
else:
    print("\n⏳ A/B Test ainda não foi executado.")
    print("   Execute: python -m evaluation.ab_test_prompts")


⏳ A/B Test ainda não foi executado.
   Execute: python -m evaluation.ab_test_prompts


## 6. Benchmark (Modelos LLM)

In [11]:
if benchmark_results:
    print("\n⚡ BENCHMARK DE MODELOS LLM\n")
    
    results_df = pd.DataFrame(benchmark_results["results"])
    display(results_df[["config_name", "model", "temperature", "faithfulness", "answer_relevancy", "latency_avg_s"]].round(3))
    
    # Trade-off
    print("\n📊 ANÁLISE TRADE-OFF\n")
    for idx, row in results_df.iterrows():
        if "error" not in row:
            print(f"  {row['config_name']:20s} | Qualidade: {row['faithfulness']:.3f} | Latência: {row['latency_avg_s']:.2f}s")
else:
    print("\n⏳ Benchmark ainda não foi executado.")
    print("   Execute: python -m evaluation.benchmark_llm")


⏳ Benchmark ainda não foi executado.
   Execute: python -m evaluation.benchmark_llm


## 7. Conclusão & Recomendações

In [12]:
print("\n" + "="*70)
print("CONCLUSÃO EXECUTIVA")
print("="*70 + "\n")

# Scoring geral
judge_avg = np.mean([judge_summary.get("averages", {}).get("factual_correctness_avg", 0),
                      judge_summary.get("averages", {}).get("business_relevance_avg", 0),
                      judge_summary.get("averages", {}).get("no_hallucination_avg", 0)])

ragas_avg = np.mean([ragas_summary.get("faithfulness", 0),
                      ragas_summary.get("answer_relevancy", 0),
                      ragas_summary.get("context_precision", 0),
                      ragas_summary.get("context_recall", 0)])

print(f"📊 Score Geral Judge:  {judge_avg:.2f}/5.0")
print(f"📊 Score Geral RAGAS:  {ragas_avg:.3f}/1.0\n")

# Interpretação
if judge_avg >= 4.0:
    readiness = "✅ PRONTO PARA PRODUÇÃO"
elif judge_avg >= 3.5:
    readiness = "⚠️  BOAS CONDIÇÕES (melhorias recomendadas)"
else:
    readiness = "❌ MELHORIAS NECESSÁRIAS"

print(f"Status: {readiness}\n")

# Recomendações automáticas
print("🎯 RECOMENDAÇÕES:\n")

recommendations = []

if judge_summary.get("averages", {}).get("factual_correctness_avg", 5) < 3.5:
    recommendations.append("1. Melhorar factual correctness — revisar dados/ferramentas")

if judge_summary.get("averages", {}).get("business_relevance_avg", 5) < 3.5:
    recommendations.append("2. Aumentar relevância para analistas — refinar prompts")

if judge_summary.get("averages", {}).get("no_hallucination_avg", 5) < 4.0:
    recommendations.append("3. Reduzir hallucinations — aumentar temperatura ou temperature tuning")

if "Judge_Média" in df_results.columns and (df_results["Judge_Média"] < 3.5).sum() > 0:
    n_bad = (df_results["Judge_Média"] < 3.5).sum()
    recommendations.append(f"4. Revisar {n_bad} perguntas com score baixo (ver seção 4)")

if ab_test_results:
    winner = ab_test_results.get("winner", "N/A")
    recommendations.append(f"5. A/B test: adotar prompt '{winner}'")

if benchmark_results:
    results_df = pd.DataFrame(benchmark_results["results"])
    best_quality = results_df.loc[results_df["faithfulness"].idxmax()]
    recommendations.append(f"6. Benchmark: considerar modelo '{best_quality['config_name']}' (melhor qualidade)")

if len(recommendations) == 0:
    recommendations.append("✅ Agente performando bem — nenhuma ação crítica necessária")

for rec in recommendations:
    print(f"  {rec}")

print("\n" + "="*70)


CONCLUSÃO EXECUTIVA

📊 Score Geral Judge:  2.73/5.0
📊 Score Geral RAGAS:  0.000/1.0

Status: ❌ MELHORIAS NECESSÁRIAS

🎯 RECOMENDAÇÕES:

  1. Melhorar factual correctness — revisar dados/ferramentas
  2. Aumentar relevância para analistas — refinar prompts
  3. Reduzir hallucinations — aumentar temperatura ou temperature tuning
  4. Revisar 12 perguntas com score baixo (ver seção 4)

